# 07 — Enhancement Round 1 (plateau check)

Three independent enhancers tested robust-simple models, careful ensembles, and error-motivated features — every candidate judged under BOTH random CV (primary) and ticket-group CV (secondary), with pre-committed adoption gates. No test labels, no downloads, no leaderboard use; all numbers re-verified below.


In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))


In [ ]:
R = json.loads((ROOT/'reports'/'experiment_results.json').read_text())
E1 = json.loads((ROOT/'reports/gap2_e1.json').read_text())
E2 = json.loads((ROOT/'reports/gap2_e2.json').read_text())
E3 = json.loads((ROOT/'reports/gap2_e3.json').read_text())
RF = R['best_cv']
print('reference RF random-CV:', round(RF, 4))

## Gate checks (fail loudly on drift)


In [ ]:
import numpy as np
assert abs(E3['base']['random']['mean'] - RF) < 1e-9, 'E3 base drifted'
assert E3['recommendation']['adopt_bool'] is False
cal = next(e for e in E1['experiments'] if e['name'] == 'calibrated_rf_cv5')
vote = next(e for e in E2['experiments'] if e['name'] == 'soft_vote_rf_gb_hgb')
d = np.array(vote['random']['folds']) - np.array([0.8603351955307262, 0.8370786516853933, 0.8202247191011236, 0.848314606741573, 0.8539325842696629])
t = float(d.mean() / (d.std(ddof=1) / np.sqrt(5)))
print('calibrated: random %.4f group %.4f brier %.4f' % (cal['random']['mean'], cal['group']['mean'], cal.get('brier', float('nan'))))
print('vote: random %.4f group %.4f | paired diff mean %+.5f t=%.2f' % (vote['random']['mean'], vote['group']['mean'], d.mean(), t))
assert abs(t) < 2.0, 'vote unexpectedly significant'
print('gates pass: enhancer numbers reproduce; vote-vs-RF separation is noise-level')

## Round-1 outcome: plateau — RF retained

- Best challengers: soft-vote RF+GB+HGB (+0.0034 random, +0.0057 group) and calibrated RF (+0.0022, Brier 0.1215 vs 0.1240).
- Both pass the pre-committed numerical gates, but the gates were necessary, not sufficient: paired vote-vs-RF t = 0.51, and the seed-noise range (0.0134) is 4–6× the candidate deltas. By the project's own standard (0.0011 treated as noise), these are indistinguishable from noise.
- Switching production model + submission on noise-level gains would be churn plus leaderboard pressure. The vote also worsens the worst fold (0.7697 vs 0.7809).
- Feature variants: all correctly rejected (best +0.0011 with worse group direction).
- Recorded as E13–E15: evidence of diminishing returns at this representation, not failures. The final model and submission are unchanged.
